# Линейная регрессия нативными инструментами python

Датасет https://insideairbnb.com/get-the-data/#:~:text=Lisbon%2C%20Lisbon%2C%20Portugal

Подготовка данных 

In [1]:
import pandas
import numpy

In [2]:
df = pandas.read_csv("data/lisbon1.csv")
df.head()


,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,6499,https://www.airbnb.com/rooms/6499,20260919032754,2026-09-20,NaN,Belém 1 Bedroom Historical Apartment,"This apartment is all about Location, next to ...",NaN,https://a0.muscache.com/pictures/hosting/Hosti...,14455,...,4.89,4.81,4.39,NaN,NaN,1,1,0,0,0.84
1,25659,https://www.airbnb.com/rooms/25659,20260919032754,2026-09-21,NaN,Lisbon City Center Apartment (Alfama),Cozy apartment in Lisbon's most historic and b...,NaN,https://a0.muscache.com/pictures/hosting/Hosti...,107347,...,4.95,4.86,4.81,56539/AL.,NaN,1,1,0,0,1.54
2,84599,https://www.airbnb.com/rooms/84599,20260919032754,2026-09-21,NaN,"Chic 2-Bed Baixa Apt | Elevator, A/C & River View",Step onto your private balcony and enjoy beaut...,NaN,https://a0.muscache.com/pictures/prohost-api/H...,181902,...,4.80,4.91,4.60,8954/AL,NaN,19,19,0,0,0.59
3,89015,https://www.airbnb.com/rooms/89015,20260919032754,2026-09-21,NaN,Lisbon Cottage - Principe Real,A unique two bedroom house right in the center...,NaN,https://a0.muscache.com/pictures/0049965b-65ec...,485573,...,4.73,4.87,4.63,88834/AL,NaN,1,1,0,0,0.82
4,91282,https://www.airbnb.com/rooms/91282,20260919032754,2026-09-21,NaN,Lovely apartment in Estrela,This is a one-bedroom apartment with a bathroo...,NaN,https://a0.muscache.com/pictures/miso/Hosting-...,133304,...,4.86,4.77,4.41,5478/AL,NaN,1,1,0,0,0.41


In [3]:
df = df[["neighbourhood_cleansed", "property_type", "room_type", "accommodates", 
"bathrooms_text", "bedrooms", "beds", "price"]]


df.head()

,neighbourhood_cleansed,property_type,room_type,accommodates,bathrooms_text,bedrooms,beds,price
0,Belm,Entire rental unit,Entire home/apt,3,1 bath,1.0,3.0,$89.00
1,Santa Maria Maior,Entire condo,Entire home/apt,3,1 bath,1.0,2.0,$120.00
2,Santa Maria Maior,Entire rental unit,Entire home/apt,5,1 bath,2.0,4.0,$440.00
3,Misericrdia,Entire home,Entire home/apt,4,1 bath,2.0,3.0,NaN
4,Campo de Ourique,Entire rental unit,Entire home/apt,2,1 bath,1.0,2.0,$80.00


In [4]:
df.shape

(25020, 8)

In [5]:
df.info()

df.isna().sum()

<class 'pandas.DataFrame'>
RangeIndex: 25020 entries, 0 to 25019
Data columns (total 8 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   neighbourhood_cleansed  25020 non-null  str    
 1   property_type           25020 non-null  str    
 2   room_type               25020 non-null  str    
 3   accommodates            25020 non-null  int64  
 4   bathrooms_text          25006 non-null  str    
 5   bedrooms                24972 non-null  float64
 6   beds                    24638 non-null  float64
 7   price                   22043 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 1.5 MB


neighbourhood_cleansed       0
property_type                0
room_type                    0
accommodates                 0
bathrooms_text              14
bedrooms                    48
beds                       382
price                     2977
dtype: int64

In [6]:
df = df.dropna()
df.info()

<class 'pandas.DataFrame'>
Index: 21656 entries, 0 to 25019
Data columns (total 8 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   neighbourhood_cleansed  21656 non-null  str    
 1   property_type           21656 non-null  str    
 2   room_type               21656 non-null  str    
 3   accommodates            21656 non-null  int64  
 4   bathrooms_text          21656 non-null  str    
 5   bedrooms                21656 non-null  float64
 6   beds                    21656 non-null  float64
 7   price                   21656 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 1.5 MB


In [7]:
other_neighbourhood_cleansed = df['neighbourhood_cleansed'].value_counts().tail(80).index

In [8]:
newcols = pandas.DataFrame()
newcols['neighbourhood_other'] =  df['neighbourhood_cleansed'].isin(other_neighbourhood_cleansed).astype(int)
for neighbourhood in df['neighbourhood_cleansed'].unique():
    if neighbourhood not in other_neighbourhood_cleansed:
        newcols['neighbourhood_' + neighbourhood] = (df["neighbourhood_cleansed"] == neighbourhood).astype(int)
df = pandas.concat([newcols, df], axis=1)    

In [9]:
df = df.drop(columns="neighbourhood_cleansed")

In [10]:
df.head()

,neighbourhood_other,neighbourhood_Belm,neighbourhood_Santa Maria Maior,neighbourhood_Campo de Ourique,neighbourhood_So Vicente,neighbourhood_Estrela,neighbourhood_Misericrdia,neighbourhood_Arroios,neighbourhood_Santo Antnio,neighbourhood_Cascais e Estoril,...,neighbourhood_Sacavm e Prior Velho,neighbourhood_Venteira,neighbourhood_Mina de gua,property_type,room_type,accommodates,bathrooms_text,bedrooms,beds,price
0,0,1,0,0,0,0,0,0,0,0,...,0,0,0,Entire rental unit,Entire home/apt,3,1 bath,1.0,3.0,$89.00
1,0,0,1,0,0,0,0,0,0,0,...,0,0,0,Entire condo,Entire home/apt,3,1 bath,1.0,2.0,$120.00
2,0,0,1,0,0,0,0,0,0,0,...,0,0,0,Entire rental unit,Entire home/apt,5,1 bath,2.0,4.0,$440.00
4,0,0,0,1,0,0,0,0,0,0,...,0,0,0,Entire rental unit,Entire home/apt,2,1 bath,1.0,2.0,$80.00
5,0,0,0,0,1,0,0,0,0,0,...,0,0,0,Entire rental unit,Entire home/apt,4,1 bath,1.0,1.0,$122.50


In [11]:
for room_type in df['room_type'].unique():
    df['room_type_' + room_type] = (df['room_type'] == room_type).astype(int)
df = df.drop(columns='room_type')

In [12]:
df['bathrooms_text'].unique()

<StringArray>
[           '1 bath',     '1 shared bath',           '2 baths',
           '3 baths',         '2.5 baths',         '1.5 baths',
           '5 baths',    '1 private bath',    '2 shared baths',
         '5.5 baths',         '4.5 baths',           '0 baths',
           '8 baths',         '3.5 baths',           '4 baths',
    '3 shared baths',         '6.5 baths',  '2.5 shared baths',
  '1.5 shared baths',  'Shared half-bath',           '6 baths',
  '3.5 shared baths',    '8 shared baths',   '10 shared baths',
    '0 shared baths',  '7.5 shared baths',           '7 baths',
    '4 shared baths',         'Half-bath',          '14 baths',
  '4.5 shared baths',          '21 baths',          '12 baths',
 'Private half-bath',          '23 baths',           '9 baths',
    '6 shared baths',  '6.5 shared baths',         '8.5 baths',
         '9.5 baths',          '10 baths',          '15 baths',
         '7.5 baths',   '14 shared baths',    '5 shared baths',
        '10.5 baths']
Leng

In [13]:
bt = df['bathrooms_text'].str.lower()
n = pandas.to_numeric(bt.str.split().str[0], errors='coerce')
n.loc[bt.str.contains('half')] = 0.5

b_shared_mask = bt.str.contains('shared') 
b_private_mask = bt.str.contains('private')

df['baths_shared'] = n.where(b_shared_mask, 0)
df['baths_private'] = n.where(b_private_mask, 0)
df['baths_plain'] = n.where(~b_shared_mask & ~b_private_mask, 0)

df = df.drop(columns='bathrooms_text')

In [14]:
property_other = df['property_type'].value_counts().tail(50).index

newcols = pandas.DataFrame()
newcols['property_other'] = df['property_type'].isin(property_other).astype(int)
for property in df['property_type'].unique():
    if property not in property_other:
        newcols['property' + property] = (df['property_type'] == property).astype(int)

df = pandas.concat([df, newcols], axis=1)

In [15]:
df = df.drop(columns='property_type')

In [16]:
df['price'] = df['price'].str.replace('$', '').str.replace(',', '').astype(float)

In [17]:
df.head().T

,0,1,2,4,5
neighbourhood_other,0.0,0.0,0.0,0.0,0.0
neighbourhood_Belm,1.0,0.0,0.0,0.0,0.0
neighbourhood_Santa Maria Maior,0.0,1.0,1.0,0.0,0.0
neighbourhood_Campo de Ourique,0.0,0.0,0.0,1.0,0.0
neighbourhood_So Vicente,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...
propertyEntire place,0.0,0.0,0.0,0.0,0.0
propertyCamper/RV,0.0,0.0,0.0,0.0,0.0
propertyPrivate room in serviced apartment,0.0,0.0,0.0,0.0,0.0
propertyShared room in guesthouse,0.0,0.0,0.0,0.0,0.0


## План:

## Класс linear_regressor

### Функция fit

Общий подход по линейной регрессии

Заводим константный признак x0 для смещения

Инициализируем вектор весов w размерностью равной l - количеству признаков, присваивая каждому w единицу

Выбираем Loss функцию (MSE) 

Делим выборку на k батчей для кросс валидации
Для каждого батча:

В цикле количества эпох:
    В цикле n/m:
        Берем батч размерностью m
        Вычисляем градиент от текущего веса:
            Берем производную от текущей линейной функции и весов

        Отнимаем из текущего веса k * градиент, где k - шаг градиента
    
Проводим валидацию на df_test

*Добавить регулиризацию elastic net

In [18]:
# Перемешиваем выборку

# Перемешиваем индексы
import random 

random.seed(42)

idxs = [i for i in range(0, len(df))]
n = len(idxs)
for i in range(n-1):
    j = random.randint(i, n-1)
    idxs[i], idxs[j] = idxs[j], idxs[i]

# Перемешиваем строки фрейма

df = df.iloc[idxs].reset_index(drop=True)

# Добавляем параметр x0 

df['const'] = 1

# Делим на train и test

test = df.iloc[:int(0.2*n)]
train = df.iloc[int(0.2*n):]



In [21]:
df.head()

,neighbourhood_other,neighbourhood_Belm,neighbourhood_Santa Maria Maior,neighbourhood_Campo de Ourique,neighbourhood_So Vicente,neighbourhood_Estrela,neighbourhood_Misericrdia,neighbourhood_Arroios,neighbourhood_Santo Antnio,neighbourhood_Cascais e Estoril,...,propertyBoat,propertyRoom in serviced apartment,propertyPrivate room in condo,propertyRoom in hotel,propertyEntire place,propertyCamper/RV,propertyPrivate room in serviced apartment,propertyShared room in guesthouse,propertyPrivate room in casa particular,const
0,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
1,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
2,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
3,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
4,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,1


In [ ]:
def fit_fold(X, Y):
    # Инициализируем вектор весов w
    length, width = X.shape[0], X.shape[1]
    w = [0] * width

    l_rate = 1
    batches = 10
    epochs = 5

    batch_size = int(length/batches)

    for i in range(epochs):
        for j in range(batches):
            batch_X = X.iloc[j*batch_size:(j+1)*batch_size]
            batch_Y = Y.iloc[j*batch_size:(j+1)*batch_size]

            for i in range(w):
                w[i] = w[i] - l_rate * (1/len(batch_X) * mse_der())

    def mse_der(X, Y, w):
        res = 0
        length, width = X.shape[0], X.shape[1]
        for i in range(length):
            y_pred = 0
            for j in range(width):
                y_pred += w[j] + X[i][j]

            res += y_pred - Y[i]

    return


In [20]:
#fit 
Y = train['price']
X = train.drop(columns='price')

# Границы фолдов

folds = 5

fold_size = int(train.shape[0]/folds)
fold_borders = [int(fold_size) * i for i in range(folds)] + [len(train)]

for i in range(folds):
    fold_test = train.iloc[fold_borders[i]:fold_borders[i+1]]
    fold_train = pandas.concat([train.iloc[:fold_borders[i]], train.iloc[fold_borders[i+1]:]])

    fold_test_X = fold_test.drop(columns='price')
    fold_test_Y = fold_test['price']

    fold_train_X = fold_train.drop(columns='price')
    fold_train_Y = fold_train['price']

    fit_fold()

TypeError: fit_fold() missing 2 required positional arguments: 'X' and 'Y'

In [ ]:
# Инициализируем вектор весов w
w = [0] * X.shape[1]


l_rate = 1
batch_size = 10

w[0] = w[0] - l_rate * (1/batch_size * mse_der())

def mse_der(X, Y, w):
    res = 0
    length, width = X.shape[0], X.shape[1]
    for i in range(length):
        y_pred = 0
        for j in range(width):
            y_pred += w[j] + X[i][j]

        res += y_pred - Y[i]


In [ ]:
for i in range(5):
    print(i)

0
1
2
3
4


In [ ]:

fold_size = int(X.shape[0]/5)
fold_borders = [int(fold_size) * i for i in range(5)] + [len(X)]

print(fold_borders)
print(len(X))

[0, 4331, 8662, 12993, 17324, 21656]
21656
